# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [30]:
from pyomo.environ import *
from pyomo.opt import SolverFactory

## 🔹 Model

In [31]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [32]:
model.COMPARTIMENTS = Set(initialize=['Front', 'Center', 'Back'])
model.CHARGES = Set(initialize=[1, 2, 3, 4])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.COMPARTIMENTS for j in model.CHARGES])

## 🔹 Parameters

In [33]:
model.cap_poids = Param(model.COMPARTIMENTS, initialize={'Front': 12.0, 'Center': 18.0, 'Back': 10.0}, within=NonNegativeReals)
model.cap_volume = Param(model.COMPARTIMENTS, initialize={'Front': 7000.0, 'Center': 9000.0, 'Back': 5000.0}, within=NonNegativeReals)
model.poids = Param(model.CHARGES, initialize={1: 20.0, 2: 16.0, 3: 25.0, 4: 13.0}, within=NonNegativeReals)
model.volume = Param(model.CHARGES, initialize={1: 500.0, 2: 700.0, 3: 600.0, 4: 400.0}, within=NonNegativeReals)
model.gain = Param(model.CHARGES, initialize={1: 320.0, 2: 400.0, 3: 360.0, 4: 290.0}, within=NonNegativeReals)

## 🔹 Variables

In [34]:
model.X = Var(model.COMPARTIMENTS, model.CHARGES, domain=NonNegativeReals)

## 🔹 Constraints

In [35]:
model.c_for_0 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_0.add(sum(model.X[c,j] for j in model.CHARGES) <= model.cap_poids[c])
model.c_for_1 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_1.add(sum(model.volume[j] * model.X[c,j] for j in model.CHARGES) <= model.cap_volume[c])
model.c_for_2 = ConstraintList()
for j in model.CHARGES:
    model.c_for_2.add(sum(model.X[c,j] for c in model.COMPARTIMENTS) <= model.poids[j])
model.c_for_3 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_3.add(sum(model.cap_poids[u] for u in model.COMPARTIMENTS) * sum(model.X[c,j] for j in model.CHARGES) == model.cap_poids[c] * sum(model.X[t,j] for t,j in model.ARC))

## 🔹 Objective

In [36]:
model.obj = Objective(expr=sum(sum(( model.gain[j] * model.X[c,j] ) for j in model.CHARGES) for c in model.COMPARTIMENTS), sense=maximize)

## ⚙️ Résolution du modèle

In [37]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('Solver status:', result.solver.status)
print('Termination condition:', result.solver.termination_condition)

Solver status: ok
Termination condition: optimal


## 📊 Valeurs optimales des variables

In [38]:
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v.name}')
    for index in v:
        print(f'   {index} = {v[index].value}')

Variable set: X
   ('Front', 1) = 0.0
   ('Front', 2) = 0.0
   ('Front', 3) = 11.0
   ('Front', 4) = 1.0
   ('Center', 1) = 0.0
   ('Center', 2) = 6.0
   ('Center', 3) = -0.0
   ('Center', 4) = 12.0
   ('Back', 1) = 10.0
   ('Back', 2) = 0.0
   ('Back', 3) = 0.0
   ('Back', 4) = -0.0


In [39]:
model.obj()

13330.0